In [10]:
import json
import pandas as pd
import numpy as np


def build_master_table(
    omission_path,
    llm_path,
    article,
    sections_path="Sections_data/controversy_score.csv"
):
    # Omission + section IDs
    with open(omission_path, encoding="utf-8") as f:
        data = json.load(f)

    df_om = pd.DataFrame([
        {
            "Section pair": x["section_pair_id"],
            "section_id_en": x["section_en"]["section_id"],
            "section_id_es": x["section_es"]["section_id"],
            "O_en→es": x["omission_metrics"]["en_to_es"]["omission"],
            "O_es→en": x["omission_metrics"]["es_to_en"]["omission"]
        }
        for x in data
    ])

    # LLM metrics
    df = pd.read_csv(llm_path)

    counts = (
        df.groupby("section_pair_id").size()
        .rename("N_APs")
        .reset_index()
        .rename(columns={"section_pair_id": "Section pair"})
    )

    df = df[(df["comparable"] == "Yes") & (df["status"] == "ok")].copy()

    df["Contr."] = df["contradiction"].map({
        "Yes": 1,
        "Unclear": 0.5,
        "No": 0
    })

    metrics = (
        df.groupby("section_pair_id")[["Contr.", "agency", "causality", "language"]]
        .mean()
        .reset_index()
        .rename(columns={
            "section_pair_id": "Section pair",
            "agency": "F_ag",
            "causality": "F_ca",
            "language": "F_la"
        })
    )

    # Controversy per section
    sections = pd.read_csv(sections_path, sep=";")
    sections = sections[sections["pair"] == article]

    def controversy(lang):
        return (
            sections[sections["lang"] == lang][
                ["section_id", "section_score", "section_score_per_token"]
            ]
            .rename(columns={
                "section_id": f"section_id_{lang}",
                "section_score": f"C_sec_{lang}",
                "section_score_per_token": f"C_sec_{lang}_per_token"
            })
        )

    # Master
    master = (
        df_om
        .merge(counts, on="Section pair", how="left")
        .merge(metrics, on="Section pair", how="left")
        .merge(controversy("en"), on="section_id_en", how="left")
        .merge(controversy("es"), on="section_id_es", how="left")
    )

    master["ΔC_sec"] = master["C_sec_en"] - master["C_sec_es"]
    master["ΔC_sec_per_token"] = (
        master["C_sec_en_per_token"] - master["C_sec_es_per_token"]
    )

    master.insert(0, "Article", article)

    return master.dropna(
        subset=["Contr.", "F_ag", "F_ca", "F_la"],
        how="all"
    )

In [11]:
data=[{"Article": "Falkland Islands",
       "Article_id": "Q9648",
              "omission_path": "Sections_data/Falkland Islands/Omission_results_qwen.json",
              "llm_path": "Sections_data/Falkland Islands/llm_metrics_predictions_qwen.csv"},

       {"Article": "Bajo Nuevo Bank",
        "Article_id":"Q1257783",
              "omission_path": "Sections_data/Bajo Nuevo Bank/Omission_results_qwen.json",
              "llm_path": "Sections_data/Bajo Nuevo Bank/llm_metrics_predictions_qwen.csv"},

        {"Article": "Geography of Puerto Rico",
         "Article_id": "Q2527656",
              "omission_path": "Sections_data/Geography of Puerto Rico/Omission_results_qwen.json",
              "llm_path": "Sections_data/Geography of Puerto Rico/llm_metrics_predictions_qwen.csv"},

        {"Article": "Guantanamo Bay Naval Base",
            "Article_id": "Q762570",
              "omission_path": "Sections_data/Guantanamo Bay Naval Base/Omission_results_qwen.json",
              "llm_path": "Sections_data/Guantanamo Bay Naval Base/llm_metrics_predictions_qwen.csv"},

        {"Article": "San Andres and Providencia",
            "Article_id": "Q26855",
              "omission_path": "Sections_data/San Andres and Providencia/Omission_results_qwen.json",
              "llm_path": "Sections_data/San Andres and Providencia/llm_metrics_predictions_qwen.csv"},

       {"Article": "Serranilla Bank",
            "Article_id": "Q1169008",
              "omission_path": "Sections_data/Serranilla Bank/Omission_results_qwen.json",
              "llm_path": "Sections_data/Serranilla Bank/llm_metrics_predictions_qwen.csv"},

       {"Article": "Status of Gibraltar",
            "Article_id": "Q169896",
              "omission_path": "Sections_data/Status of Gibraltar/Omission_results_qwen.json",
              "llm_path": "Sections_data/Status of Gibraltar/llm_metrics_predictions_qwen.csv"}

      ]

In [12]:
df_master_list = []
for entry in data:
    master_table = build_master_table(
        omission_path=entry["omission_path"],
        llm_path=entry["llm_path"],
        article=entry["Article"]
    )
    df_master_list.append(master_table)

df_master = pd.concat(df_master_list, ignore_index=True)
df_master.head(20)

,Article,Section pair,section_id_en,section_id_es,O_en→es,O_es→en,N_APs,Contr.,F_ag,F_ca,F_la,C_sec_en,C_sec_en_per_token,C_sec_es,C_sec_es_per_token,ΔC_sec,ΔC_sec_per_token
0,Falkland Islands,en:Q9648:0+es:Q9648:0+Best Sentence Median,en:Q9648:0,es:Q9648:0,0.735849,0.523810,25.0,0.104167,0.333333,0.250000,0.958333,293.975326,0.763572,83.309925,0.329288,210.665401,0.434284
1,Falkland Islands,en:Q9648:1+es:Q9648:1+Best Sentence Median,en:Q9648:1,es:Q9648:1,0.642857,0.272727,17.0,0.437500,0.000000,0.125000,0.500000,118.542316,0.415938,42.572678,0.417379,75.969638,-0.001441
2,Falkland Islands,en:Q9648:2+es:Q9648:2+Best Sentence Median,en:Q9648:2,es:Q9648:2,0.897959,0.836957,22.0,0.595238,0.666667,1.095238,1.190476,803.876683,0.531664,120.061856,0.103680,683.814827,0.427984
3,Falkland Islands,en:Q9648:3+es:Q9648:3+Best Sentence Median,en:Q9648:3,es:Q9648:3,0.786885,0.644444,27.0,0.166667,0.583333,0.500000,0.708333,377.692914,0.648957,44.580425,0.107682,333.112489,0.541275
4,Falkland Islands,en:Q9648:6+es:Q9648:5+Best Sentence Median,en:Q9648:6,es:Q9648:5,0.985075,0.894737,2.0,0.500000,2.000000,2.000000,2.000000,193.963993,0.313351,2.912223,0.023298,191.051770,0.290053
5,Falkland Islands,en:Q9648:7+es:Q9648:6+Best Sentence Median,en:Q9648:7,es:Q9648:6,0.382979,0.333333,45.0,0.321429,0.619048,0.619048,0.880952,169.586780,0.471074,33.944103,0.080246,135.642677,0.390828
6,Falkland Islands,en:Q9648:9+es:Q9648:7+Best Sentence Median,en:Q9648:9,es:Q9648:7,0.956522,0.977273,1.0,1.000000,2.000000,2.000000,2.000000,48.931609,0.253532,37.636066,0.080591,11.295543,0.172940
7,Bajo Nuevo Bank,en:Q1257783:0+es:Q1257783:0+Best Sentence Median,en:Q1257783:0,es:Q1257783:0,0.642857,0.500000,9.0,0.312500,0.250000,0.250000,0.375000,16.156759,0.183600,9.041383,0.062354,7.115376,0.121245
8,Bajo Nuevo Bank,en:Q1257783:1+es:Q1257783:1+Best Sentence Median,en:Q1257783:1,es:Q1257783:1,0.725000,0.083333,31.0,0.483333,0.666667,0.466667,1.333333,13.023333,0.053594,5.333333,0.042328,7.690000,0.011266
9,Bajo Nuevo Bank,en:Q1257783:2+es:Q1257783:2+Best Sentence Median,en:Q1257783:2,es:Q1257783:2,0.758621,0.176471,29.0,0.583333,0.333333,0.333333,0.333333,69.869907,0.109343,9.871315,0.042185,59.998592,0.067157
